In [1]:
import tensorflow as tf

if tf.config.list_physical_devices('GPU'):
    print("GPU is available!")
    print(tf.config.list_physical_devices('GPU'))
else:
    print("GPU is not available. Please check runtime settings.")

GPU is not available. Please check runtime settings.


In [2]:
import os
import pandas as pd
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np
from concurrent.futures import ThreadPoolExecutor
import joblib
from sklearn.utils import shuffle

In [3]:
# Đường dẫn đến dữ liệu trên Kaggle
base_dir = '/kaggle/input/cs114-all-cars/'
csv_dir = '/kaggle/input/split-data-car'
save_dir = '/kaggle/working/'

# Thay đổi đoạn code tải model
weights_path = '/kaggle/input/pretrained-weights/mobilenet_v2_weights_tf_dim_ordering_tf_kernels_1.0_224_no_top.h5'
model = MobileNetV2(weights=weights_path, include_top=False, input_shape=(224, 224, 3))

# Bản đồ từ tên hiệu xe sang CategoryID
category_map = {
    "Others": 0,
    "Honda": 1,
    "Hyundai": 2,
    "KIA": 3,
    "Mazda": 4,
    "Mitsubishi": 5,
    "Suzuki": 6,
    "Toyota": 7,
    "VinFast": 8
}

In [4]:
# Trích xuất đặc trưng từ MobileNet
def extract_features(df, split_name):
    features_file = os.path.join(save_dir, f"Features_{split_name}_Split_{split_index}.npz")

    # Nếu tệp đã tồn tại, tải lại
    if os.path.exists(features_file):
        print(f"Loading features from {features_file}")
        data = np.load(features_file)
        return data['features'], data['labels']

    features, labels = [], []

    def process_image(row):
        try:
            image_path = os.path.join(base_dir, row['ImageFullPath'])
            label = category_map[row['ImageFullPath'].split('/')[0]]
            
            # Load và tiền xử lý ảnh
            image = load_img(image_path, target_size=(224, 224))
            image_array = img_to_array(image) / 255.0
            feature = model.predict(np.expand_dims(image_array, axis=0))
            features.append(feature.flatten())
            labels.append(label)
        except Exception as e:
            print(f"Error loading image {row['ImageFullPath']}: {e}")

    with ThreadPoolExecutor() as executor:
        executor.map(process_image, [row for _, row in df.iterrows()])

    # Lưu đặc trưng
    np.savez(features_file, features=np.array(features), labels=np.array(labels))
    print(f"Features saved to {features_file}")
    return np.array(features), np.array(labels)

In [5]:
# Số lượng split cần xử lý
num_splits = 5

# Lặp qua từng split
for split_index in range(1, num_splits + 1):
    if (split_index == 1):
        # Đường dẫn đến file đặc trưng
        features_file_train = os.path.join('/kaggle/input/split1-extract-features/Features_Train_Split_1.npz')
        features_file_test = os.path.join('/kaggle/input/split1-extract-features/Features_Test_Split_1.npz')
        # Tải dữ liệu
        train_data = np.load(features_file_train)
        X_train, y_train = train_data['features'], train_data['labels']
        test_data = np.load(features_file_test)
        X_test, y_test = test_data['features'], test_data['labels']
    if (split_index == 2):
        # Đường dẫn đến file đặc trưng
        features_file_train = os.path.join('/kaggle/input/split23-extract-features/Features_Train_Split_2.npz')
        features_file_test = os.path.join('/kaggle/input/split23-extract-features/Features_Test_Split_2.npz')
        # Tải dữ liệu
        train_data = np.load(features_file_train)
        X_train, y_train = train_data['features'], train_data['labels']
        test_data = np.load(features_file_test)
        X_test, y_test = test_data['features'], test_data['labels']
    if (split_index == 3):
        # Đường dẫn đến file đặc trưng
        features_file_train = os.path.join('/kaggle/input/split23-extract-features/Features_Train_Split_3.npz')
        features_file_test = os.path.join('/kaggle/input/split23-extract-features/Features_Test_Split_3.npz')
        # Tải dữ liệu
        train_data = np.load(features_file_train)
        X_train, y_train = train_data['features'], train_data['labels']
        test_data = np.load(features_file_test)
        X_test, y_test = test_data['features'], test_data['labels']
    if (split_index == 4):
        # Đường dẫn đến file đặc trưng
        features_file_train = os.path.join('/kaggle/input/split45-extract-features/Features_Train_Split_4.npz')
        features_file_test = os.path.join('/kaggle/input/split45-extract-features/Features_Test_Split_4.npz')
        # Tải dữ liệu
        train_data = np.load(features_file_train)
        X_train, y_train = train_data['features'], train_data['labels']
        test_data = np.load(features_file_test)
        X_test, y_test = test_data['features'], test_data['labels']
    if (split_index == 5):
        # Đường dẫn đến file đặc trưng
        features_file_train = os.path.join('/kaggle/input/split45-extract-features/Features_Train_Split_5.npz')
        features_file_test = os.path.join('/kaggle/input/split45-extract-features/Features_Test_Split_5.npz')
        # Tải dữ liệu
        train_data = np.load(features_file_train)
        X_train, y_train = train_data['features'], train_data['labels']
        test_data = np.load(features_file_test)
        X_test, y_test = test_data['features'], test_data['labels']
    

    # train_path = os.path.join(csv_dir, f"CarDataset-Splits-{split_index}-Train.csv")
    # test_path = os.path.join(csv_dir, f"CarDataset-Splits-{split_index}-Test.csv")

    # # Đọc dữ liệu Train và Test
    # train_df = pd.read_csv(train_path)
    # test_df = pd.read_csv(test_path)

    # X_train, y_train = extract_features(train_df, "Train")
    # X_test, y_test = extract_features(test_df, "Test")

    # Thiết lập Random Forest
    classifier = RandomForestClassifier()

    # Huấn luyện mô hình
    classifier.fit(X_train, y_train)

    # Đánh giá trên tập test
    accuracy = classifier.score(X_test, y_test)
    y_pred = classifier.predict(X_test)
    conf_matrix = confusion_matrix(y_test, y_pred)
    class_report = classification_report(y_test, y_pred)

    # In kết quả
    print(f"Split {split_index}: Final Accuracy: {accuracy:.4f}")
    print(f"Confusion Matrix:\n{conf_matrix}")
    print(f"Classification Report:\n{class_report}")

    # Lưu kết quả vào file riêng cho từng split
    result_path = os.path.join(save_dir, f"Results_Split_{split_index}.txt")
    with open(result_path, 'w') as file:
        file.write(f"Split {split_index}: Final Accuracy: {accuracy:.4f}\n\n")
        file.write(f"Confusion Matrix:\n{conf_matrix}\n\n")
        file.write(f"Classification Report:\n{class_report}")

    # Lưu mô hình của từng split
    model_path = os.path.join(save_dir, f"Model_Split_{split_index}.joblib")
    joblib.dump(classifier, model_path)
    print(f"Split {split_index} model saved to {model_path}")

Split 1: Final Accuracy: 0.4070
Confusion Matrix:
[[471  13  47  15  38   7 139 177   2]
 [106 111  35   8  23   8 131 197   6]
 [160  15 166   9  25   6 157 141   6]
 [140  12  26 181  28   3 122 124   2]
 [176  14  23   8 181   8  85 134   5]
 [130  13  27  16  19  50 155 157   5]
 [ 76   8  15   5  17   6 975 202   7]
 [134  13  29   8  49   7 301 607   6]
 [145   7  17  10  25   6 100 109 144]]
Classification Report:
              precision    recall  f1-score   support

           0       0.31      0.52      0.38       909
           1       0.54      0.18      0.27       625
           2       0.43      0.24      0.31       685
           3       0.70      0.28      0.40       638
           4       0.45      0.29      0.35       634
           5       0.50      0.09      0.15       572
           6       0.45      0.74      0.56      1311
           7       0.33      0.53      0.40      1154
           8       0.79      0.26      0.39       563

    accuracy                     